In [ ]:
"""
Dog Robot — Pure NN + PPO  (ΔΔθ actions — acceleration-level control)
=================================================================================
Curriculum variant: DELTA_HIP and DELTA_KNEE grow linearly from START to END
over N_EP episodes.  Each episode the joint limits are recomputed so the policy
starts with a tight range of motion and gradually unlocks the full ±1 rad range.

  ep=0         : Δhip=DELTA_HIP_START,  Δknee=DELTA_KNEE_START
  ep=N_EP-1    : Δhip=DELTA_HIP_END=1,  Δknee=DELTA_KNEE_END=1

Everything else (PPO+GAE, ΔΔθ control, ESP32 export) is unchanged.
"""

# ── Hyper-parameters ───────────────────────────────────────────
N_EP         = 10000
ALIGN_W      = 0.00
YAW_W        = 0.20
ALIVE_BONUS  = 0.05

MAX_DDELTA   = 0.02          # rad per step — acceleration limit (ΔΔθ scale)
DELTA_LIMIT  = 0.05          # rad per step — cap on delta buffer

ACTION_REG_W = 0.0001
DELTA_REG_W  = 0.0002
JVEL_W       = 0.05
JEXC_W       = 0.3

# ── Velocity tracking ──────────────────────────────────────────
TARGET_VX   = 1.0 / 4.0
VX_TRACK_W  = 4.0
VX_TOL      = 0.02

import os, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import mujoco
import mediapy as media
import matplotlib.pyplot as plt
from tqdm import trange

os.makedirs("movies",       exist_ok=True)
os.makedirs("models",       exist_ok=True)
os.makedirs("plots",        exist_ok=True)
os.makedirs("trajectories", exist_ok=True)

# ── Robot geometry ─────────────────────────────────────────────
HIP_INIT  =  0.90
KNEE_INIT = -1.40

# ── Curriculum: joint half-range grows from START → END ────────
DELTA_HIP_START  = 0.40   # initial hip  half-range (rad)
DELTA_KNEE_START = 0.20   # initial knee half-range (rad)
DELTA_HIP_END    = .60   # final   hip  half-range (rad)
DELTA_KNEE_END   = .40   # final   knee half-range (rad)

# Full-range limits used for eval rollouts and ESP32 export
JOINT_LO_FINAL = np.array([HIP_INIT  - DELTA_HIP_END,
                            KNEE_INIT - DELTA_KNEE_END] * 4, dtype=np.float32)
JOINT_HI_FINAL = np.array([HIP_INIT  + DELTA_HIP_END,
                            KNEE_INIT + DELTA_KNEE_END] * 4, dtype=np.float32)
OFFSETS        = np.array([HIP_INIT, KNEE_INIT] * 4, dtype=np.float32)
# globals kept for export helpers — overwritten to FINAL before export
JOINT_LO = JOINT_LO_FINAL.copy()
JOINT_HI = JOINT_HI_FINAL.copy()

SUBSTEPS    = 5
SETTLE_TIME = 0.3
EPISODE_DUR = 8.0

# ── PPO hyper-parameters ──────────────────────────────────────
LR           = 3e-4
GAMMA        = 0.99
GAE_LAMBDA   = 0.95
CLIP_EPS     = 0.2
ENTROPY_COEF = 0.005
VF_COEF      = 0.5
MAX_GRAD     = 0.5

N_STEPS_PER_UPDATE = 4096
PPO_EPOCHS         = 10
MINIBATCH_SIZE     = 256

LOG_STD_INIT = 0.5
LOG_STD_MIN  = -2.0
LOG_STD_MAX  = 1.0

RENDER_EVERY = 200

LOAD_MODEL = ""  # e.g. "models/nn_ppo_ddtheta_best.pth"
EVAL_ONLY  = False

# ── Dimensions ─────────────────────────────────────────────────
ACTION_DIM = 8
HIDDEN_DIM = 64
STATE_DIM  = 24

mjmodel = mujoco.MjModel.from_xml_path("dog.xml")
mjdata  = mujoco.MjData(mjmodel)
CTRL_DT = mjmodel.opt.timestep * SUBSTEPS
print(f"CTRL_DT = {CTRL_DT*1000:.1f} ms  ({1/CTRL_DT:.0f} Hz)")
print(f"State dim = {STATE_DIM}  |  Action dim = {ACTION_DIM}")
print(f"MAX_DDELTA = {MAX_DDELTA} rad/step  |  DELTA_LIMIT = {DELTA_LIMIT} rad/step")
print(f"Curriculum: Δhip {DELTA_HIP_START}→{DELTA_HIP_END}  "
      f"Δknee {DELTA_KNEE_START}→{DELTA_KNEE_END} rad over {N_EP} episodes")


# ══════════════════════════════════════════════════════════════
#  Curriculum helper
# ══════════════════════════════════════════════════════════════
def curriculum_ranges(ep: int):
    """Joint limits expand linearly with episode number.
    Returns (joint_lo, joint_hi, delta_hip, delta_knee)."""
    frac = np.clip((ep - START_EP) / max(END_EP - START_EP - 1, 1), 0.0, 1.0)
    dh   = DELTA_HIP_START  + frac * (DELTA_HIP_END  - DELTA_HIP_START)
    dk   = DELTA_KNEE_START + frac * (DELTA_KNEE_END - DELTA_KNEE_START)
    jlo  = np.array([HIP_INIT - dh, KNEE_INIT - dk] * 4, dtype=np.float32)
    jhi  = np.array([HIP_INIT + dh, KNEE_INIT + dk] * 4, dtype=np.float32)
    return jlo, jhi, dh, dk


# ══════════════════════════════════════════════════════════════
#  Helpers
# ══════════════════════════════════════════════════════════════
def orientation():
    qw, qx, qy, qz = mjdata.qpos[3:7]
    roll  = math.atan2(2*(qw*qx + qy*qz), 1 - 2*(qx**2 + qy**2))
    pitch = math.asin(float(np.clip(2*(qw*qy - qz*qx), -1, 1)))
    yaw   = math.atan2(2*(qw*qz + qx*qy), 1 - 2*(qy**2 + qz**2))
    return roll, pitch, yaw


def build_state(delta: np.ndarray) -> np.ndarray:
    """24-dim: jpos(8) + jvel(8) + delta(8)."""
    return np.concatenate([
        mjdata.qpos[7:15].astype(np.float32),
        mjdata.qvel[6:14].astype(np.float32),
        delta.astype(np.float32),
    ])


def apply_ddelta(ctrl: np.ndarray, delta: np.ndarray, ddelta: np.ndarray,
                 joint_lo: np.ndarray, joint_hi: np.ndarray):
    """Two-level integration with episode-specific joint limits."""
    delta_new = np.clip(delta + ddelta * MAX_DDELTA, -DELTA_LIMIT, DELTA_LIMIT)
    ctrl_new  = np.clip(ctrl  + delta_new, joint_lo, joint_hi)
    return ctrl_new, delta_new


def is_done():
    _, p, _ = orientation()
    return mjdata.qpos[2] < 0.03 or abs(p) > math.radians(45)


def reset_sim():
    ip = np.array([HIP_INIT, KNEE_INIT] * 4, dtype=np.float32)
    mujoco.mj_resetData(mjmodel, mjdata)
    mjdata.qpos[7:15] = ip
    mjdata.qpos[2]    = 0.10
    mujoco.mj_forward(mjmodel, mjdata)
    for _ in range(int(SETTLE_TIME / mjmodel.opt.timestep)):
        mjdata.ctrl[:] = ip
        mujoco.mj_step(mjmodel, mjdata)
    delta = np.zeros(ACTION_DIM, dtype=np.float32)
    return ip.copy(), delta


# ══════════════════════════════════════════════════════════════
#  Actor-Critic
# ══════════════════════════════════════════════════════════════
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.pi = nn.Sequential(
            nn.Linear(STATE_DIM,  HIDDEN_DIM), nn.ReLU(),
            nn.Linear(HIDDEN_DIM, ACTION_DIM),
        )
        nn.init.orthogonal_(self.pi[-1].weight, gain=0.01)
        nn.init.zeros_(self.pi[-1].bias)
        self.log_std = nn.Parameter(torch.full((ACTION_DIM,), LOG_STD_INIT))
        self.vf = nn.Sequential(
            nn.Linear(STATE_DIM, 256), nn.ReLU(),
            nn.Linear(256, 256),       nn.ReLU(),
            nn.Linear(256, 1),
        )
        nn.init.orthogonal_(self.vf[-1].weight, gain=1.0)
        nn.init.zeros_(self.vf[-1].bias)

    def forward_pi(self, s):
        mean    = self.pi(s)
        log_std = self.log_std.clamp(LOG_STD_MIN, LOG_STD_MAX)
        return mean, log_std

    def forward_vf(self, s):
        return self.vf(s).squeeze(-1)

    def get_action(self, s, deterministic=False):
        mean, log_std = self.forward_pi(s)
        std = log_std.exp()
        raw = mean if deterministic else mean + std * torch.randn_like(std)
        action = torch.tanh(raw)
        log_prob = (-0.5 * ((raw - mean) / (std + 1e-8))**2
                    - log_std - 0.5 * math.log(2 * math.pi))
        log_prob -= torch.log(1 - action.pow(2) + 1e-6)
        log_prob  = log_prob.sum(-1)
        value = self.forward_vf(s)
        return action, log_prob, value

    def evaluate_actions(self, s, actions_tanh):
        mean, log_std = self.forward_pi(s)
        std = log_std.exp()
        raw = torch.atanh(actions_tanh.clamp(-0.999, 0.999))
        log_prob = (-0.5 * ((raw - mean) / (std + 1e-8))**2
                    - log_std - 0.5 * math.log(2 * math.pi))
        log_prob -= torch.log(1 - actions_tanh.pow(2) + 1e-6)
        log_prob  = log_prob.sum(-1)
        entropy = (log_std + 0.5 * math.log(2 * math.pi * math.e)).sum(-1)
        value   = self.forward_vf(s)
        return log_prob, entropy, value


ac  = ActorCritic().to(device)
opt = torch.optim.Adam(ac.parameters(), lr=LR, eps=1e-5)


# ══════════════════════════════════════════════════════════════
#  Checkpoint loading
# ══════════════════════════════════════════════════════════════
if LOAD_MODEL:
    print(f"Loading checkpoint: {LOAD_MODEL}")
    ck = torch.load(LOAD_MODEL, map_location=device)
    ac.load_state_dict(ck.get("best_w") or ck["ac"])
    if "opt" in ck:
        opt.load_state_dict(ck["opt"])
    best_dist_loaded = ck.get("best_dist", -np.inf)
    ep_loaded        = ck.get("ep", 0)
    rl   = ck.get("rl",   [])
    dl   = ck.get("dl",   [])
    cll  = ck.get("cll",  [])
    all_ = ck.get("all_", [])
    print(f"  Resumed ep={ep_loaded}  best_dist={best_dist_loaded:.3f}m")
else:
    best_dist_loaded = -np.inf
    ep_loaded        = 0
    rl, dl, cll, all_ = [], [], [], []


# ══════════════════════════════════════════════════════════════
#  PPO buffer
# ══════════════════════════════════════════════════════════════
class PPOBuffer:
    def __init__(self):
        self.clear()

    def clear(self):
        self.states    = []
        self.actions   = []
        self.log_probs = []
        self.rewards   = []
        self.values    = []
        self.dones     = []

    def push(self, s, a, lp, r, v, d):
        self.states.append(s)
        self.actions.append(a)
        self.log_probs.append(lp)
        self.rewards.append(r)
        self.values.append(v)
        self.dones.append(d)

    def __len__(self):
        return len(self.states)

    def compute_gae(self, last_value):
        T       = len(self.rewards)
        adv     = np.zeros(T, dtype=np.float32)
        ret     = np.zeros(T, dtype=np.float32)
        values  = np.array(self.values + [last_value], dtype=np.float32)
        rewards = np.array(self.rewards, dtype=np.float32)
        dones   = np.array(self.dones,   dtype=np.float32)
        gae = 0.0
        for t in reversed(range(T)):
            delta  = rewards[t] + GAMMA * values[t+1] * (1-dones[t]) - values[t]
            gae    = delta + GAMMA * GAE_LAMBDA * (1-dones[t]) * gae
            adv[t] = gae
            ret[t] = adv[t] + values[t]
        return ret, adv

    def get_tensors(self, last_value):
        ret, adv = self.compute_gae(last_value)
        S   = torch.FloatTensor(np.array(self.states)).to(device)
        A   = torch.FloatTensor(np.array(self.actions)).to(device)
        LP  = torch.FloatTensor(np.array(self.log_probs)).to(device)
        R   = torch.FloatTensor(ret).to(device)
        Adv = torch.FloatTensor(adv).to(device)
        Adv = (Adv - Adv.mean()) / (Adv.std() + 1e-8)
        return S, A, LP, R, Adv


ppo_buf = PPOBuffer()


# ══════════════════════════════════════════════════════════════
#  PPO update
# ══════════════════════════════════════════════════════════════
def update_ppo(last_value):
    S, A, LP_old, R, Adv = ppo_buf.get_tensors(last_value)
    N = S.shape[0]
    total_pi_loss = 0.0
    total_vf_loss = 0.0
    n_updates     = 0
    for _ in range(PPO_EPOCHS):
        idx = torch.randperm(N, device=device)
        for start in range(0, N, MINIBATCH_SIZE):
            mb     = idx[start : start + MINIBATCH_SIZE]
            lp_new, entropy, v_new = ac.evaluate_actions(S[mb], A[mb])
            ratio   = (lp_new - LP_old[mb]).exp()
            surr1   = ratio * Adv[mb]
            surr2   = ratio.clamp(1-CLIP_EPS, 1+CLIP_EPS) * Adv[mb]
            pi_loss = -torch.min(surr1, surr2).mean()
            vf_loss = F.mse_loss(v_new, R[mb])
            loss = pi_loss + VF_COEF * vf_loss - ENTROPY_COEF * entropy.mean()
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(ac.parameters(), MAX_GRAD)
            opt.step()
            total_pi_loss += pi_loss.item()
            total_vf_loss += vf_loss.item()
            n_updates     += 1
    ppo_buf.clear()
    return total_vf_loss / max(n_updates, 1), total_pi_loss / max(n_updates, 1)


# ══════════════════════════════════════════════════════════════
#  Training rollout  (receives current-episode joint limits)
# ══════════════════════════════════════════════════════════════
def rollout_train(joint_lo, joint_hi):
    ctrl, delta = reset_sim()
    x0          = mjdata.qpos[0]
    _, _, yaw0  = orientation()
    tot_r       = 0.0
    s           = build_state(delta)

    done = False
    for t in range(int(EPISODE_DUR / CTRL_DT)):
        st = torch.FloatTensor(s).unsqueeze(0).to(device)
        with torch.no_grad():
            action, log_prob, value = ac.get_action(st, deterministic=False)
        an = action.squeeze(0).cpu().numpy()
        lp = log_prob.item()
        v  = value.item()

        ctrl, delta = apply_ddelta(ctrl, delta, an, joint_lo, joint_hi)
        mjdata.ctrl[:] = ctrl
        for _ in range(SUBSTEPS):
            mujoco.mj_step(mjmodel, mjdata)

        roll, pitch, yaw = orientation()
        dyaw      = (yaw - yaw0 + math.pi) % (2 * math.pi) - math.pi
        vx        = float(mjdata.qvel[0])
        vy        = abs(float(mjdata.qvel[1]))
        jvel_mag  = float(np.mean(np.abs(mjdata.qvel[6:14])))
        jpos      = mjdata.qpos[7:15].astype(np.float32)
        excursion = float(np.mean(np.abs(jpos - OFFSETS)))

        vx_err = max(0.0, abs(vx - TARGET_VX) - VX_TOL)
        vx_rew = -VX_TRACK_W * vx_err ** 2

        r = (vx_rew
             + ALIVE_BONUS
             - 0.10 * vy
             - ACTION_REG_W * float(np.sum(an**2))
             - DELTA_REG_W  * float(np.sum(delta**2))
             + JVEL_W  * jvel_mag
             + JEXC_W  * excursion
             - ALIGN_W * (roll**2 + pitch**2)
             - YAW_W   * dyaw**2)
        tot_r += r
        done = is_done()
        if done:
            r -= 5.0

        ppo_buf.push(s, an, lp, r, v, float(done))
        s = build_state(delta)
        if done:
            break

    if done:
        last_val = 0.0
    else:
        with torch.no_grad():
            st = torch.FloatTensor(s).unsqueeze(0).to(device)
            last_val = ac.forward_vf(st).item()

    dist = mjdata.qpos[0] - x0
    return tot_r, dist, last_val


# ══════════════════════════════════════════════════════════════
#  Eval rollout  (deterministic, uses full joint range by default)
# ══════════════════════════════════════════════════════════════
def rollout_eval(render=False, fps=60, record_traj=False,
                 joint_lo=None, joint_hi=None):
    if joint_lo is None:
        joint_lo = JOINT_LO_FINAL
    if joint_hi is None:
        joint_hi = JOINT_HI_FINAL

    ctrl, delta = reset_sim()
    x0     = mjdata.qpos[0]
    tot_r  = 0.0
    frames = []
    traj_ctrl  = []
    traj_delta = []
    renderer = None

    if render:
        renderer = mujoco.Renderer(mjmodel, height=1080, width=1920)
        cam = mujoco.MjvCamera()
        mujoco.mjv_defaultFreeCamera(mjmodel, cam)
        cam.distance  = 0.6
        cam.elevation = -20
        fe = max(1, int(1 / (fps * CTRL_DT)))

    s = build_state(delta)

    for t in range(int(EPISODE_DUR / CTRL_DT)):
        with torch.no_grad():
            st = torch.FloatTensor(s).unsqueeze(0).to(device)
            action, _, _ = ac.get_action(st, deterministic=True)
            an = action.squeeze(0).cpu().numpy()

        ctrl, delta = apply_ddelta(ctrl, delta, an, joint_lo, joint_hi)
        mjdata.ctrl[:] = ctrl
        for _ in range(SUBSTEPS):
            mujoco.mj_step(mjmodel, mjdata)

        if record_traj:
            traj_ctrl.append(ctrl.copy())
            traj_delta.append(delta.copy())

        vx     = float(mjdata.qvel[0])
        vx_err = max(0.0, abs(vx - TARGET_VX) - VX_TOL)
        tot_r += -VX_TRACK_W * vx_err ** 2
        done = is_done()
        if done:
            tot_r -= 5.0

        s = build_state(delta)

        if render and t % fe == 0:
            cam.lookat[0] = mjdata.qpos[0]
            cam.lookat[1] = mjdata.qpos[1]
            cam.azimuth   = (cam.azimuth + 0.2) % 360
            renderer.update_scene(mjdata, cam)
            frames.append(renderer.render().copy())

        if done:
            break

    dist = mjdata.qpos[0] - x0
    if renderer:
        renderer.close()
    if record_traj:
        return tot_r, dist, frames, np.array(traj_ctrl, dtype=np.float32), np.array(traj_delta, dtype=np.float32)
    if render:
        return tot_r, dist, frames
    return tot_r, dist


# ══════════════════════════════════════════════════════════════
#  Trajectory export
# ══════════════════════════════════════════════════════════════
JOINT_NAMES = [
    "FL_hip", "FL_knee", "FR_hip", "FR_knee",
    "RL_hip", "RL_knee", "RR_hip", "RR_knee",
]


def export_trajectory(traj: np.ndarray, tag: str = "best", delta_traj: np.ndarray = None):
    T    = traj.shape[0]
    base = f"trajectories/{tag}"

    np.save(f"{base}.npy", traj)
    print(f"  Saved {base}.npy  shape={traj.shape}")

    if delta_traj is not None:
        np.save(f"{base}_delta.npy", delta_traj)
        print(f"  Saved {base}_delta.npy  (delta buffer evolution)")

    header = "step_ms," + ",".join(JOINT_NAMES)
    rows = [f"{i*CTRL_DT*1000:.2f}," + ",".join(f"{v:.6f}" for v in row)
            for i, row in enumerate(traj)]
    with open(f"{base}.csv", "w") as f:
        f.write(header + "\n" + "\n".join(rows) + "\n")
    print(f"  Saved {base}.csv  ({T} rows)")

    scale    = 10_000
    traj_rel = traj - OFFSETS[np.newaxis, :]
    int16    = np.clip(np.round(traj_rel * scale), -32768, 32767).astype(np.int16)

    max_abs = np.abs(traj_rel[0]).max()
    if max_abs > 0.05:
        print(f"  Warning: step-0 max |offset| = {max_abs:.4f} rad")

    lines = [
        f"/* Auto-generated trajectory  tag={tag}",
        f" * Steps: {T}  dt: {CTRL_DT*1000:.2f} ms  ({1/CTRL_DT:.0f} Hz)",
        f" * Joints: {', '.join(JOINT_NAMES)}",
        f" * Control scheme: delta-delta (ΔΔθ) — acceleration-level actions",
        f" * VALUES ARE RELATIVE TO NEUTRAL POSE (OFFSETS subtracted):",
        f" * neutral = [{', '.join(f'{v:.4f}' for v in OFFSETS)}] rad",
        f" * angle_rad  = OFFSET[j] + traj[step][j] / {scale}.0f",
        f" * pulse      = CENTER    + traj[step][j] / {scale}.0f * PULSE_PER_RAD * DIR[ch]",
        " */",
        "#pragma once", "#include <stdint.h>", "",
        f"#define TRAJ_STEPS  {T}",
        f"#define TRAJ_JOINTS 8",
        f"#define TRAJ_DT_MS  {int(round(CTRL_DT * 1000))}",
        f"#define TRAJ_SCALE  {scale}",
        f"/* neutral offsets (rad×TRAJ_SCALE) for reference:",
        f"   HIP_INIT  = {int(round(HIP_INIT  * scale))}",
        f"   KNEE_INIT = {int(round(KNEE_INIT * scale))} */", "",
        f"static const int16_t traj[{T}][8] = {{",
    ]
    for i, row in enumerate(int16):
        vals  = ", ".join(f"{int(v):6d}" for v in row)
        comma = "," if i < T - 1 else " "
        lines.append(f"  {{ {vals} }}{comma}  /* t={i*CTRL_DT*1000:7.1f} ms */")
    lines.append("};");
    with open(f"{base}.h", "w") as f:
        f.write("\n".join(lines) + "\n")
    print(f"  Saved {base}.h  ({T*8*2/1024:.1f} KB)")

    t_axis = np.arange(T) * CTRL_DT
    ncols  = 3 if delta_traj is not None else 2
    fig, axes = plt.subplots(4, ncols, figsize=(6*ncols, 10), sharex=True)
    for idx, name in enumerate(JOINT_NAMES):
        ax_pos = axes[idx // 2, idx % 2]
        ax_pos.plot(t_axis, traj[:, idx], lw=1.2)
        ax_pos.set_title(f"{name} pos", fontsize=9)
        ax_pos.set_ylabel("rad", fontsize=7)
        ax_pos.grid(True, alpha=0.3)
    if delta_traj is not None:
        for idx, name in enumerate(JOINT_NAMES):
            axes[idx // 2, 2].plot(t_axis, delta_traj[:, idx], lw=1.0, alpha=0.7, label=name)
        for r in range(4):
            axes[r, 2].legend(fontsize=6)
            axes[r, 2].set_title("delta buffer", fontsize=9)
            axes[r, 2].grid(True, alpha=0.3)
    for c in range(ncols):
        axes[-1, c].set_xlabel("time (s)")
    plt.suptitle(f"Trajectory (ΔΔθ) — {T} steps @ {1/CTRL_DT:.0f} Hz", fontsize=11)
    plt.tight_layout()
    plt.savefig(f"{base}_plot.png", dpi=120)
    plt.close()
    print(f"  Saved {base}_plot.png")


# ══════════════════════════════════════════════════════════════
#  ESP32 policy export
# ══════════════════════════════════════════════════════════════
def export_policy_header(path: str = "models/policy.h"):
    sd = ac.state_dict()
    W1 = sd["pi.0.weight"].cpu().numpy()
    b1 = sd["pi.0.bias"  ].cpu().numpy()
    W2 = sd["pi.2.weight"].cpu().numpy()
    b2 = sd["pi.2.bias"  ].cpu().numpy()
    H  = W1.shape[0]; S = W1.shape[1]; A = W2.shape[0]

    def arr1d(name, vec):
        vals = ", ".join(f"{v:.8f}f" for v in vec.flatten())
        return f"static const float {name}[{vec.size}] = {{{vals}}};\n"

    def arr2d(name, mat, comment=""):
        rows, cols = mat.shape
        out = ([f"/* {comment} */"] if comment else []) + [
            f"static const float {name}[{rows}][{cols}] = {{"]
        for i, row in enumerate(mat):
            vals  = ", ".join(f"{v:.8f}f" for v in row)
            comma = "," if i < rows - 1 else ""
            out.append(f"  {{ {vals} }}{comma}")
        out.append("};"); return "\n".join(out) + "\n\n"

    def flt(name, val, comment=""):
        c = f"  /* {comment} */" if comment else ""
        return f"static const float {name} = {val:.8f}f;{c}\n"

    lines = [f"""\
#pragma once
/*
 * Auto-generated policy header — DO NOT EDIT BY HAND
 *
 * Architecture : {S}-input  ->  ReLU({H})  ->  tanh({A})-output
 * Training     : PPO+GAE  delta-delta (acceleration-level) control
 *                Curriculum: DELTA_HIP/KNEE {DELTA_HIP_START}→{DELTA_HIP_END} rad
 * Target speed : {TARGET_VX:.4f} m/s  ({1.0/TARGET_VX:.1f} s per metre)
 * Control dt   : {CTRL_DT*1000:.2f} ms  ({1/CTRL_DT:.0f} Hz)
 */

#include <math.h>
#include <string.h>

#define POLICY_STATE_DIM  {S}
#define POLICY_HIDDEN_DIM {H}
#define POLICY_ACTION_DIM {A}
#define POLICY_CTRL_DT_MS {int(round(CTRL_DT * 1000))}
#define POLICY_HIP_INIT   {HIP_INIT:.6f}f
#define POLICY_KNEE_INIT  {KNEE_INIT:.6f}f

"""
    ]
    lines.append(arr2d("POLICY_W1", W1, f"W1[{H}][{S}]"))
    lines.append(arr1d("POLICY_b1", b1))
    lines.append(arr2d("POLICY_W2", W2, f"W2[{A}][{H}]"))
    lines.append(arr1d("POLICY_b2", b2))
    lines.append("\n")
    lines.append(flt("POLICY_MAX_DDELTA",  MAX_DDELTA,  "acceleration scale (rad/step)"))
    lines.append(flt("POLICY_DELTA_LIMIT", DELTA_LIMIT, "max |delta| (rad/step)"))
    lines.append(flt("POLICY_TARGET_VX",   TARGET_VX,   "desired forward speed (m/s)"))
    lines.append(f"\nstatic const float POLICY_JOINT_LO[{A}] = {{"
                 + ", ".join(f"{v:.6f}f" for v in JOINT_LO_FINAL) + "};\n")
    lines.append(f"static const float POLICY_JOINT_HI[{A}] = {{"
                 + ", ".join(f"{v:.6f}f" for v in JOINT_HI_FINAL) + "};\n")
    lines.append(f"static const float POLICY_OFFSETS[{A}]   = {{"
                 + ", ".join(f"{v:.6f}f" for v in OFFSETS)         + "};\n")
    lines.append(f"""
static inline void policy_step(
    const float state[POLICY_STATE_DIM],
    float       ctrl [POLICY_ACTION_DIM],
    float       delta[POLICY_ACTION_DIM])
{{
    int i, j, k;
    float h[POLICY_HIDDEN_DIM];
    for (i = 0; i < POLICY_HIDDEN_DIM; i++) {{
        float acc = POLICY_b1[i];
        for (j = 0; j < POLICY_STATE_DIM; j++) acc += POLICY_W1[i][j] * state[j];
        h[i] = (acc > 0.0f) ? acc : 0.0f;
    }}
    float action[POLICY_ACTION_DIM];
    for (k = 0; k < POLICY_ACTION_DIM; k++) {{
        float acc = POLICY_b2[k];
        for (i = 0; i < POLICY_HIDDEN_DIM; i++) acc += POLICY_W2[k][i] * h[i];
        action[k] = tanhf(acc);
    }}
    for (k = 0; k < POLICY_ACTION_DIM; k++) {{
        delta[k] += action[k] * POLICY_MAX_DDELTA;
        if      (delta[k] >  POLICY_DELTA_LIMIT) delta[k] =  POLICY_DELTA_LIMIT;
        else if (delta[k] < -POLICY_DELTA_LIMIT) delta[k] = -POLICY_DELTA_LIMIT;
        ctrl[k] += delta[k];
        if      (ctrl[k] > POLICY_JOINT_HI[k]) ctrl[k] = POLICY_JOINT_HI[k];
        else if (ctrl[k] < POLICY_JOINT_LO[k]) ctrl[k] = POLICY_JOINT_LO[k];
    }}
}}
""")
    with open(path, "w") as f:
        f.writelines(lines)
    total_params = H*S + H + A*H + A
    print(f"  Saved {path}")
    print(f"  Actor: W1({H}x{S}) + b1({H}) + W2({A}x{H}) + b2({A}) "
          f"= {total_params} floats = {total_params*4/1024:.1f} KB")


# ══════════════════════════════════════════════════════════════
#  Eval-only mode
# ══════════════════════════════════════════════════════════════
if EVAL_ONLY:
    assert LOAD_MODEL
    print("Eval mode...")
    _, bd, frames, traj, delta_traj = rollout_eval(render=True, record_traj=True)
    if frames:
        media.write_video("movies/nn_ppo_ddtheta_eval.mp4", frames, fps=60)
        print(f"Saved: movies/nn_ppo_ddtheta_eval.mp4  dist={bd:.3f}m")
    export_trajectory(traj, tag="eval", delta_traj=delta_traj)
    export_policy_header("models/policy_eval.h")
    import sys; sys.exit(0)


# ══════════════════════════════════════════════════════════════
#  Training loop
# ══════════════════════════════════════════════════════════════
best_dist = best_dist_loaded
best_w    = None
nm        = 0
ppo_update_count = 0

if LOAD_MODEL and best_dist_loaded > -np.inf:
    best_w = {k: v.clone() for k, v in ac.state_dict().items()}

START_EP = ep_loaded + 1 if LOAD_MODEL else 0
END_EP   = START_EP + N_EP

print(f"\nNN+PPO+GAE (ΔΔθ curriculum) | ep {START_EP}→{END_EP} "
      f"| state={STATE_DIM} act={ACTION_DIM}"
      f"| Δhip {DELTA_HIP_START}→{DELTA_HIP_END}  Δknee {DELTA_KNEE_START}→{DELTA_KNEE_END}\n")

for ep in trange(START_EP, END_EP):
    jlo, jhi, dh, dk = curriculum_ranges(ep)

    r, d, last_val = rollout_train(jlo, jhi)
    rl.append(r); dl.append(d)

    if d > best_dist:
        best_dist = d
        best_w = {k: v.clone() for k, v in ac.state_dict().items()}

    cl = al = 0.0
    if len(ppo_buf) >= N_STEPS_PER_UPDATE:
        cl, al = update_ppo(last_val)
        ppo_update_count += 1
    cll.append(cl); all_.append(al)

    if ep % 10 == 0:
        log_std_val = ac.log_std.data.mean().item()
        print(f"  ep {ep:5d} | dist={d:.3f}m avg10={np.mean(dl[-10:]):.3f} | "
              f"R={r:.2f} | buf={len(ppo_buf)} | "
              f"σ={math.exp(log_std_val):.3f} | VL={cl:.4f} PL={al:.4f} | "
              f"Δhip={dh:.3f} Δknee={dk:.3f} | updates={ppo_update_count}")

    if ep % RENDER_EVERY == 0 and ep > START_EP and best_w:
        train_w = {k: v.clone() for k, v in ac.state_dict().items()}
        ac.load_state_dict(best_w)
        _, bd, frames = rollout_eval(render=True)
        if frames:
            out = f"movies/nn_ppo_ddtheta_{nm:03d}_ep{ep:04d}_{bd:.2f}m.mp4"
            media.write_video(out, frames, fps=60)
            print(f"  → {out}")
        nm += 1
        ac.load_state_dict(train_w)

    if ep % 50 == 49:
        torch.save({
            "ac": ac.state_dict(), "opt": opt.state_dict(),
            "best_w": best_w,
            "best_dist": best_dist, "ep": ep,
            "rl": rl, "dl": dl, "cll": cll, "all_": all_,
        }, f"models/nn_ppo_ddtheta_ep{ep}.pth")

        w  = 20
        sm = lambda x: np.convolve(x, np.ones(w)/w, "valid")
        if len(dl) >= w:
            fig, ax = plt.subplots(2, 2, figsize=(12, 8))
            ax[0, 0].plot(sm(dl),  color="#2ecc71"); ax[0, 0].set_title("Distance (m)")
            ax[0, 1].plot(sm(rl),  color="#3498db"); ax[0, 1].set_title("Reward")
            ax[1, 0].plot(sm(cll), color="#e74c3c"); ax[1, 0].set_title("Value Loss")
            ax[1, 1].plot(sm(all_),color="#f39c12"); ax[1, 1].set_title("Policy Loss")
            for a_ in ax.flat:
                a_.grid(True, alpha=0.3); a_.set_xlabel("ep")
            plt.suptitle(f"NN+PPO+GAE (ΔΔθ curriculum)  ep {ep} | best={best_dist:.3f}m")
            plt.tight_layout()
            plt.savefig(f"plots/nn_ppo_ddtheta_ep{ep}.png", dpi=120)
            plt.close()


CTRL_DT = 10.0 ms  (100 Hz)
State dim = 24  |  Action dim = 8
MAX_DDELTA = 0.02 rad/step  |  DELTA_LIMIT = 0.05 rad/step
Curriculum: Δhip 0.4→0.8  Δknee 0.2→0.6 rad over 10000 episodes
Device: cpu

NN+PPO+GAE (ΔΔθ curriculum) | ep 0→10000 | state=24 act=8| Δhip 0.4→0.8  Δknee 0.2→0.6



  0%|          | 1/10000 [00:00<38:02,  4.38it/s]

  ep     0 | dist=0.148m avg10=0.148 | R=-92.60 | buf=800 | σ=1.649 | VL=0.0000 PL=0.0000 | Δhip=0.400 Δknee=0.200 | updates=0


  0%|          | 11/10000 [00:02<36:02,  4.62it/s]

  ep    10 | dist=0.090m avg10=0.141 | R=-92.63 | buf=4000 | σ=1.611 | VL=0.0000 PL=0.0000 | Δhip=0.400 Δknee=0.200 | updates=1


...

  ep  5970 | dist=1.310m avg10=0.932 | R=138.31 | buf=3200 | σ=0.690 | VL=0.0000 PL=0.0000 | Δhip=0.639 Δknee=0.439 | updates=985


 60%|█████▉    | 5982/10000 [31:32<15:11,  4.41it/s]

  ep  5980 | dist=1.197m avg10=1.294 | R=88.42 | buf=1600 | σ=0.692 | VL=0.0000 PL=0.0000 | Δhip=0.639 Δknee=0.439 | updates=987


 60%|█████▉    | 5987/10000 [31:34<21:09,  3.16it/s]


KeyboardInterrupt: 

In [ ]:
# ══════════════════════════════════════════════════════════════
#  Final render + trajectory export
# ══════════════════════════════════════════════════════════════
print(f"\n=== Best dist: {best_dist:.3f}m ===")

if best_w:
    ac.load_state_dict(best_w)
    torch.save({"ac": best_w, "best_dist": best_dist},
               "models/nn_ppo_ddtheta_best.pth")

    print("\nRecording final video + exporting trajectory...")
    _, bd, frames, traj, delta_traj = rollout_eval(render=True, record_traj=True)
    if frames:
        media.write_video("movies/nn_ppo_ddtheta_final.mp4", frames, fps=60)
        print(f"Final: movies/nn_ppo_ddtheta_final.mp4  dist={bd:.3f}m")
    export_trajectory(traj, tag="best", delta_traj=delta_traj)
    export_policy_header("models/policy.h")

    # ── One-cycle detection ───────────────────────────────────
    one_cycle_found = False
    T = traj.shape[0]
    if T > 64:
        sig   = traj[:, 0] - traj[:, 0].mean()
        fft   = np.fft.rfft(sig)
        freqs = np.fft.rfftfreq(T, d=CTRL_DT)
        dom_f = freqs[np.argmax(np.abs(fft[1:])) + 1]
        if dom_f > 0.1:
            spc = int(round(1.0 / (dom_f * CTRL_DT)))
            if T >= 2 * spc:
                export_trajectory(traj[:spc], tag="one_cycle", delta_traj=delta_traj[:spc])
                print(f"\nOne-cycle: {spc} steps ({1/dom_f:.2f}s, {dom_f:.2f} Hz)")
                one_cycle_found = True

    import shutil
    os.makedirs("export", exist_ok=True)
    shutil.copy("models/policy.h", "export/policy.h")
    print("\nESP32 export bundle → export/")
    print("  Copied: models/policy.h  →  export/policy.h")

    if one_cycle_found:
        src_traj = "trajectories/one_cycle.h"; label = "one_cycle"
    else:
        src_traj = "trajectories/best.h";      label = "best"
    shutil.copy(src_traj, "export/trajectory.h")
    print(f"  Copied: {src_traj}  →  export/trajectory.h  ({label})")

    print("\n  Sketch folder contents should be:")
    print("    albert_trajectory/")
    print("      albert_trajectory.ino")
    print("      trajectory.h          ← from export/trajectory.h")
    print("    albert_firmware/")
    print("      albert_firmware.ino")
    print("      policy.h              ← from export/policy.h")
